# 08 - Padrões de Rebaixamento

A partir de qual rodada o Z4 se define? Se um time está no Z4 na rodada X, qual a probabilidade de cair?

In [1]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df.dropna(subset=["rodata"])
df["rodata"] = df["rodata"].astype(int)
df = df[df["ano"] >= 2003].copy()

In [2]:
def classificacao_rodada_a_rodada(df_season):
    """Retorna dict: rodada -> lista ordenada de times (1º ao último)."""
    teams = set(df_season["mandante"].unique()) | set(df_season["visitante"].unique())
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    
    max_rodada = df_season["rodata"].max()
    classificacoes = {}
    
    for rodada in range(1, max_rodada + 1):
        jogos = df_season[df_season["rodata"] == rodada]
        for _, jogo in jogos.iterrows():
            m, v = jogo["mandante"], jogo["visitante"]
            gm, gv = jogo["mandante_Placar"], jogo["visitante_Placar"]
            if pd.isna(gm) or pd.isna(gv):
                continue
            gm, gv = int(gm), int(gv)
            saldo[m] += gm - gv
            saldo[v] += gv - gm
            if gm > gv:
                pontos[m] += 3; vitorias[m] += 1
            elif gm < gv:
                pontos[v] += 3; vitorias[v] += 1
            else:
                pontos[m] += 1; pontos[v] += 1
        
        ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
        classificacoes[rodada] = ranking
    
    return classificacoes, max_rodada

In [3]:
# Para cada temporada e rodada, verificar se os times no Z4 acabaram rebaixados
resultados = []

for ano in sorted(df["ano"].unique()):
    df_s = df[df["ano"] == ano]
    classif, max_rod = classificacao_rodada_a_rodada(df_s)
    if max_rod < 30:
        continue
    
    n_times = len(classif[max_rod])
    # Rebaixados = últimos 4 na rodada final
    rebaixados = set(classif[max_rod][-4:])
    
    for rodada in range(1, max_rod + 1):
        z4_rodada = set(classif[rodada][-4:])
        # Quantos dos que estão no Z4 nesta rodada acabaram rebaixados?
        acertou = len(z4_rodada & rebaixados)
        resultados.append({
            "ano": ano, "rodada": rodada,
            "pct_z4_final": acertou / 4 * 100
        })

df_reb = pd.DataFrame(resultados)

In [4]:
# Média: % dos times no Z4 na rodada X que acabaram rebaixados
prob_z4 = df_reb.groupby("rodada")["pct_z4_final"].mean().reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prob_z4["rodada"], y=prob_z4["pct_z4_final"],
    mode="lines+markers",
    line=dict(color="#e74c3c", width=3),
    marker=dict(size=5),
    hovertemplate="Rodada %{x}<br>%{y:.1f}% dos times no Z4 foram rebaixados<extra></extra>"
))

fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=75, line_dash="dash", line_color="orange", opacity=0.3,
              annotation_text="75%", annotation_position="right")

fig.update_layout(
    title="Se Está no Z4 na Rodada X, Qual a Chance de Ser Rebaixado?<br><sup>Brasileirão Série A (2003-presente)</sup>",
    xaxis_title="Rodada",
    yaxis_title="% dos Times no Z4 que Foram Rebaixados",
    yaxis=dict(range=[0, 105]),
    template="plotly_white",
    width=900,
    height=500
)

fig.show()
_path = "../charts/rebaixamento.html"
fig.write_html(_path, include_plotlyjs="cdn")

# Adiciona descrição estatística
_desc = "Tendência crescente da probabilidade de rebaixamento conforme a rodada avança, com estabilização após a rodada 25. A partir da rodada 30, mais de 75% dos times no Z4 acabam rebaixados, indicando um ponto de não-retorno estatístico."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/rebaixamento.html")

Gráfico exportado: charts/rebaixamento.html


In [5]:
# Insight
rodada_75 = prob_z4[prob_z4["pct_z4_final"] >= 75]["rodada"].min()
print(f"A partir da rodada {rodada_75}, 75%+ dos times no Z4 acabam rebaixados")

# Grandes clubes que foram rebaixados
GRANDES = ["Flamengo", "Palmeiras", "Corinthians", "Sao Paulo", "Santos",
           "Gremio", "Internacional", "Cruzeiro", "Atletico-MG",
           "Fluminense", "Vasco", "Botafogo"]

for ano in sorted(df["ano"].unique()):
    df_s = df[df["ano"] == ano]
    classif, max_rod = classificacao_rodada_a_rodada(df_s)
    if max_rod < 30:
        continue
    rebaixados = classif[max_rod][-4:]
    grandes_reb = [t for t in rebaixados if t in GRANDES]
    if grandes_reb:
        print(f"  {ano}: {', '.join(grandes_reb)}")

A partir da rodada 27, 75%+ dos times no Z4 acabam rebaixados
  2003: Fluminense
  2004: Gremio
  2005: Atletico-MG


  2007: Corinthians


  2008: Vasco
  2012: Palmeiras


  2013: Fluminense, Vasco


  2015: Vasco
  2016: Internacional


  2019: Cruzeiro
  2021: Vasco
  2023: Santos
